# Experiment 3 — Anamnesis Mapping

**Goal**: Run the anamnesis protocol on passages spanning all four books of the Wake.
For each passage, collect novel connections — graph paths activated by the bare passage
fingerprint that are *not* foregrounded by any current lens. Cluster novel connections
across passages to discover candidate new lenses.

**The PKD formulation**: The text as prompt, the model's activation pattern as retrieval target.
What does this passage *want* to be read by? Not what a given lens extracts from it — but what
query state the passage itself implicitly demands.

**Success criterion**: Novel connections across 20+ passages cluster into 2-3 coherent groups,
each definable as a new lens framework not in the initial set.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from collections import defaultdict
import json

In [ ]:
# ── Passages spanning all four books ────────────────────────────────────────
# Format: passage text, page, line, book (1-4)
PASSAGES = [
    # Book I
    {"passage": "riverrun, past Eve and Adam's, from swerve of shore to bend of bay", "page": 3, "line": 1, "book": 1},
    {"passage": "commodius vicus of recirculation back to Howth Castle and Environs.", "page": 3, "line": 2, "book": 1},
    {"passage": "The fall (bababadalgharaghtakamminarronnkonnbronntonnerronntuonnthunntrovarrhounawnskawntoohoohoordenenthurnuk!)", "page": 3, "line": 15, "book": 1},
    {"passage": "what clashes here of wills gen wonts, oystrygods gaggin fishygods!", "page": 4, "line": 1, "book": 1},
    {"passage": "Rot a peck of pa's malt had Jhem or Shen brewed by arclight", "page": 6, "line": 13, "book": 1},
    # Book II
    {"passage": "Every evening at lighting up o'clock sharp and until further notice", "page": 219, "line": 1, "book": 2},
    {"passage": "as sure as herself pits hen to paper and there's scribings scrawled", "page": 301, "line": 14, "book": 2},
    # Book III
    {"passage": "Hark! Tolv two elf kater ten (it can't be) sax.", "page": 403, "line": 17, "book": 3},
    {"passage": "And it's old and old it's sad and old it's sad and weary", "page": 407, "line": 15, "book": 3},
    {"passage": "tell me all about Anna Livia! I want to hear all", "page": 196, "line": 1, "book": 1},
    # Book IV
    {"passage": "Sandhyas! Sandhyas! Sandhyas!", "page": 593, "line": 1, "book": 4},
    {"passage": "A way a lone a last a loved a long the", "page": 628, "line": 15, "book": 4},
    {"passage": "Soft morning, city! Lsp! I am leafy speafing.", "page": 619, "line": 20, "book": 4},
    {"passage": "Mememormee! Till thousandsthee.", "page": 628, "line": 14, "book": 4},
    {"passage": "Lps. The keys to. Given!", "page": 628, "line": 16, "book": 4},
]

print(f'{len(PASSAGES)} passages across {len(set(p["book"] for p in PASSAGES))} books')

In [ ]:
# ── Run anamnesis protocol ──────────────────────────────────────────────────
USE_REAL_MODEL = False
USE_GRAPH = False  # Set True if Neo4j running with populated data

if USE_REAL_MODEL:
    from engine.model.wake_model import WakeModel
    from engine.anamnesis.protocol import AnamnesisProtocol
    from engine.tokenizer.wake_tokenizer import WakeTokenizer
    from lenses.registry import get_lens

    model = WakeModel.from_pretrained('meta-llama/Llama-3.1-8B')
    tokenizer = WakeTokenizer('engine/tokenizer/morpheme_db/seed_morphemes.json', graph_client=None)

    # Pre-compute lens centroids (normally from training data; use random for dev)
    lens_names = ['viconian', 'kabbalistic', 'freudian', 'irish_mythology', 'norse', 'brunian']
    lens_embeddings = {name: np.random.randn(model.d_model) for name in lens_names}

    protocol = AnamnesisProtocol(model, graph_client=None, probes={}, lens_embeddings=lens_embeddings)

    anamnesis_results = []
    for p in PASSAGES:
        wake_tokens = tokenizer.tokenize_passage(p['passage'], p['page'], p['line'])
        result = protocol.run(p['passage'], wake_tokens)
        anamnesis_results.append({'passage_meta': p, 'result': result})

else:
    # Synthetic anamnesis results for offline exploration
    np.random.seed(7)
    lens_names = ['viconian', 'kabbalistic', 'freudian', 'irish_mythology', 'norse', 'brunian']

    # Synthetic novel connection types — some will cluster
    NOVEL_TYPES = [
        'astronomical+cyclic',    # candidate: hermetic/alchemical lens
        'acoustic+liturgical',    # candidate: musical/liturgical lens
        'topographic+body',       # candidate: somatic geography lens
        'number+letter',          # candidate: gematria/numerological lens
        'geological+temporal',    # candidate: deep-time / Vichian geology lens
    ]

    anamnesis_results = []
    for p in PASSAGES:
        # Simulate: each passage activates 1-3 novel connection types
        n_novel = np.random.randint(1, 4)
        novel = []
        for _ in range(n_novel):
            ntype = np.random.choice(NOVEL_TYPES, p=[0.30, 0.25, 0.20, 0.15, 0.10])
            novel.append({
                'from': p['passage'].split()[0],
                'to': p['passage'].split()[-1],
                'path': ntype,
                'novelty_score': float(np.random.beta(2, 1.5)),
            })

        # Simulate lens similarities
        lens_sims = sorted(
            [(name, float(np.random.beta(2, 3))) for name in lens_names],
            key=lambda x: -x[1]
        )

        anamnesis_results.append({
            'passage_meta': p,
            'result': {
                'retrieved_lenses': lens_sims,
                'novel_connections': novel,
                'narrative': f'Passage activates {lens_sims[0][0]} ({lens_sims[0][1]:.2f}) and {lens_sims[1][0]} ({lens_sims[1][1]:.2f}). {n_novel} novel connections.',
            }
        })

print(f'Collected anamnesis results for {len(anamnesis_results)} passages')

In [ ]:
# ── Aggregate novel connections ──────────────────────────────────────────────
novel_path_counts = defaultdict(list)

for ar in anamnesis_results:
    result = ar['result'] if isinstance(ar['result'], dict) else ar['result'].__dict__
    novel_conns = result.get('novel_connections', [])
    if hasattr(novel_conns, '__iter__'):
        for nc in novel_conns:
            if isinstance(nc, dict):
                path_type = nc.get('path', 'unknown')
                score = nc.get('novelty_score', 0.5)
                novel_path_counts[path_type].append({
                    'passage': ar['passage_meta']['passage'][:50],
                    'book': ar['passage_meta']['book'],
                    'score': score,
                })

df_novel = pd.DataFrame([
    {'path_type': pt, 'count': len(entries), 'mean_score': np.mean([e['score'] for e in entries]),
     'books': sorted(set(e['book'] for e in entries))}
    for pt, entries in novel_path_counts.items()
]).sort_values('count', ascending=False)

print('Novel connection types found:')
print(df_novel.to_string(index=False))

In [ ]:
# ── Visualise: lens similarity heatmap across passages ──────────────────────
similarity_matrix = np.zeros((len(PASSAGES), len(lens_names)))

for i, ar in enumerate(anamnesis_results):
    result = ar['result'] if isinstance(ar['result'], dict) else ar['result'].__dict__
    sims = dict(result.get('retrieved_lenses', []))
    for j, lname in enumerate(lens_names):
        similarity_matrix[i, j] = sims.get(lname, 0.0)

passage_labels = [f"p{p['page']} B{p['book']}: {p['passage'][:30]}..." for p in PASSAGES]

fig = go.Figure(data=go.Heatmap(
    z=similarity_matrix,
    x=lens_names,
    y=passage_labels,
    colorscale='Viridis',
    hoverongaps=False,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Anamnesis: Lens Similarity per Passage',
    paper_bgcolor='#0d1117',
    plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    xaxis=dict(title='Lens'),
    yaxis=dict(title='Passage', autorange='reversed'),
    height=600, width=900,
    margin=dict(l=350)
)

fig.show()
fig.write_html('e3_lens_similarity.html')
print('Saved e3_lens_similarity.html')

In [ ]:
# ── Novel connection clustering ──────────────────────────────────────────────
# Build a passage × path_type co-occurrence matrix and cluster with UMAP
all_path_types = list(novel_path_counts.keys())
passage_vectors = np.zeros((len(PASSAGES), max(len(all_path_types), 1)))

for i, ar in enumerate(anamnesis_results):
    result = ar['result'] if isinstance(ar['result'], dict) else ar['result'].__dict__
    for nc in result.get('novel_connections', []):
        if isinstance(nc, dict):
            pt = nc.get('path', '')
            if pt in all_path_types:
                j = all_path_types.index(pt)
                passage_vectors[i, j] += nc.get('novelty_score', 0.5)

# Dimensionality reduction
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42, min_dist=0.3)
    coords_2d = reducer.fit_transform(passage_vectors)
    method = 'UMAP'
except ImportError:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    coords_2d = pca.fit_transform(passage_vectors)
    method = 'PCA'

book_colors = {1: '#58a6ff', 2: '#3fb950', 3: '#d29922', 4: '#f78166'}
colors = [book_colors[p['book']] for p in PASSAGES]

fig = go.Figure(data=go.Scatter(
    x=coords_2d[:, 0], y=coords_2d[:, 1],
    mode='markers+text',
    marker=dict(size=12, color=colors, line=dict(width=1, color='white')),
    text=[f"B{p['book']}p{p['page']}" for p in PASSAGES],
    textposition='top center',
    hovertext=[f"Book {p['book']}, p.{p['page']}: {p['passage'][:60]}" for p in PASSAGES],
))

# Legend traces
for book, color in book_colors.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=10, color=color),
        name=f'Book {book}'
    ))

fig.update_layout(
    title=f'Novel Connection Clustering ({method}) — Candidate New Lenses',
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    legend=dict(bgcolor='#21262d'),
    xaxis=dict(title=f'{method} 1', gridcolor='#21262d'),
    yaxis=dict(title=f'{method} 2', gridcolor='#21262d'),
    width=900, height=600
)

fig.show()
fig.write_html('e3_novel_clusters.html')
print('Saved e3_novel_clusters.html')

In [ ]:
# ── Report candidate new lenses ──────────────────────────────────────────────
if len(df_novel) > 0:
    print('\n=== CANDIDATE NEW LENSES ===')
    print('(Path types appearing in ≥3 passages with mean novelty >0.5)\n')

    candidates = df_novel[(df_novel['count'] >= 2) & (df_novel['mean_score'] > 0.4)]

    for _, row in candidates.iterrows():
        print(f"Candidate: {row['path_type']}")
        print(f"  Occurrences: {row['count']} passages, mean novelty {row['mean_score']:.3f}")
        print(f"  Books: {row['books']}")
        print(f"  Definition: This path type connects semantic fields not addressed")
        print(f"  by Viconian, Kabbalistic, Freudian, Irish, Norse, or Brunian lenses.")
        print(f"  → Register as new Lens with foregrounded_fields=[{repr(row['path_type'])}]")
        print()
else:
    print('No novel connections found — run with USE_REAL_MODEL=True and USE_GRAPH=True')

## Interpretation

Passages that cluster together in the novel-connection space are being 'reached for' by the
same unknown framework. If the cluster is coherent — if the path types make sense as a unified
interpretive strategy — that cluster defines a candidate new lens.

For example: if many passages activate `astronomical+cyclic` connections that none of the
six current lenses foreground, this suggests an **alchemical/hermetic lens** is needed —
one that foregrounds planetary cycles, metal-to-gold transmutation, the Rosicrucian tradition,
and Bruno's own hermetic sources (Corpus Hermeticum, Ficino).

These candidate lenses are the primary scholarly contribution of the WAKE project:
new reading frameworks derived empirically from model internals, not from prior human scholarship.

**Next steps**:
1. Implement candidate lenses in `lenses/`
2. Rerun E2 with expanded lens set
3. Check whether superposition scores increase (more frames = more superposition)
4. Run E4 (Four-Headed Shin) to test PaRDeS hermeneutic depth structure